# 15 — Coefficient Monte Carlo Sensitivity Analysis

**Purpose:** Quantify how uncertainty in the EES marginal-effect coefficients propagates into
resilience score uncertainty, and identify which coefficients are both high-leverage and
weakly sourced — i.e., where empirical validation effort should go first.

**Engine:** TERRA v4.1 (`mw_action_library_v3.json`, schema 3.3)  
**Notebook:** Run in two modes — `QUICK_TEST = True` (200 iterations, ~1 min) to validate
the approach, then `QUICK_TEST = False` (10,000 iterations, overnight) for the full run.

---
**Outputs:**
- `data/processed/mc_representative_portfolio.json`
- `data/processed/mc_results_checkpoint.parquet` (incremental; survives kernel crashes)
- `data/processed/mc_validation_priority.csv`
- `data/processed/figures/mc_tornado_{composite,E,Ec,S}.png`
- Updated `data/processed/network_metadata.json` with `mc_sensitivity` block

## Executive Summary

> **[Run the quick-test cells first, then update this section with real numbers.]**
>
> Under current coefficient uncertainty (±50% at 95% CI), the composite resilience score
> for the 30-action representative portfolio is **X.XX ± Y.YY** (90% interval: [A, B]).
>
> The 5 highest-leverage coefficients are: **[fill after run]**.
> Of these, **N** are currently sourced at low-confidence (proxy / expert estimate), meaning
> validation of those N coefficients would most reduce decision uncertainty.
>
> Coefficient uncertainty **[does / does not]** change the ranking of sub-portfolios
> (environmental-heavy vs. economic-heavy vs. balanced) in **[X]%** of draws.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys, json, copy, os, time, warnings
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from scipy.stats import truncnorm, spearmanr
import matplotlib
matplotlib.use('Agg')   # non-interactive backend safe for notebooks
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore', category=FutureWarning)

# ── Project paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd()
# Tolerate running from project root or a subdirectory
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR  = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR  = DATA_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import terra_engine as te

# ── Named constants ───────────────────────────────────────────────────────────
QUICK_TEST           = False      # <-- Full run (10,000 iterations)
N_ITERATIONS         = 10_000     # Full-run iteration count
N_QUICK              = 200        # Quick-test iteration count
COEFF_CI_HALF_WIDTH  = 0.5        # ±50% uncertainty band
COEFF_CI_Z           = 1.96       # Z for 95% CI  →  SD = HALF_WIDTH / Z ≈ 0.255
CHECKPOINT_INTERVAL  = 500        # Write parquet every N completed iterations
N_WORKERS            = max(1, (os.cpu_count() or 2) - 1)
SUB_PORTFOLIO_N      = 1_000      # Paired iterations for sub-portfolio comparison

N_ITER = N_QUICK if QUICK_TEST else N_ITERATIONS
DO_CHECKPOINT = not QUICK_TEST

CHECKPOINT_PATH = DATA_DIR / 'mc_results_checkpoint.parquet'
SIGMA = COEFF_CI_HALF_WIDTH / COEFF_CI_Z

print(f"Mode         : {'QUICK TEST (' + str(N_ITER) + ' iters, no checkpoint)' if QUICK_TEST else 'FULL RUN (' + str(N_ITER) + ' iters, checkpoint every ' + str(CHECKPOINT_INTERVAL) + ')'}")
print(f"Workers      : {N_WORKERS}")
print(f"SD (sigma)   : {SIGMA:.4f}  (CI half-width={COEFF_CI_HALF_WIDTH}, Z={COEFF_CI_Z})")
print(f"Truncation   : [0.5, 1.5]  (={1-COEFF_CI_HALF_WIDTH:.1f} to {1+COEFF_CI_HALF_WIDTH:.1f})")

In [ ]:
# ── Load action library & confirm which version is active ─────────────────────
# network_metadata.json does not yet exist; v3.3 is current per mw_action_library_v3.json.
lib_path = DATA_DIR / 'mw_action_library_v3.json'
with open(lib_path) as f:
    BASE_LIBRARY = json.load(f)

schema_ver = BASE_LIBRARY.get('schema_version') or BASE_LIBRARY.get('metadata', {}).get('schema_version')
print(f"Action library: {lib_path.name}  |  schema_version={schema_ver}  |  {len(BASE_LIBRARY['actions'])} actions")

# ── Initialize base state (template — never mutated after this cell) ──────────
print("\nInitializing base state …")
t0 = time.time()
BASE_STATE = te.initialize_state()
print(f"Base state ready in {time.time()-t0:.1f}s  |  {len(BASE_STATE['county_ees'])} counties")

# ── Confirm function signatures at runtime ─────────────────────────────────────
import inspect
for fn in (te.initialize_state, te.apply_action, te.compute_ees_summary):
    print(f"  {fn.__name__}{inspect.signature(fn)}")

## Representative Portfolio

A fixed 30-action portfolio spanning all capital tiers and every major bucket in the action library.
**This portfolio is held constant across all Monte Carlo iterations; only the coefficients vary.**
The goal is not to find the *optimal* portfolio — it is to provide a stable target for sensitivity
analysis so that coefficient influence can be isolated from portfolio-selection effects.

### Selection rationale

| Bucket | Actions included | Rationale |
|---|---|---|
| `energy_generation` | wind, solar, geothermal, coal_repowering, smr_advanced, coal_to_solar, battery_grid | Covers dominant MW pathways across Mountain West BAs; all have non-trivial Ec coefficients |
| `energy_transmission` | transmission_230kv, microgrid | Grid backbone + distributed resilience; different Ec/S profiles |
| `hydrological_restoration` | riparian_buffer, beaver_reintroduction, wetland_restoration, watershed_protection, mine_land_reclamation, floodplain_reconnection | E-dominant actions; tests whether ecological coefficients drive score variance |
| `terrestrial_ecosystem` | prairie_restoration, invasive_treatment, forest_restoration, sagebrush_restoration | High E coefficients (up to 1.26); largest single-action E leverage in library |
| `economic_development` | clean_manufacturing | High Ec, near-zero E/S — isolates economic coefficient sensitivity |
| `settlement_social` | university_research_center, tribal_energy_sovereignty, rural_broadband, health_clinic, workforce_retraining, affordable_housing, lead_service_line | Covers full range of S coefficients (0.05 → 0.61) across different subpopulations |
| `agriculture` | irrigation_efficiency, rangeland_restoration_maintenance | Wyoming-specific; tests whether small-county ag actions shift study-area aggregate |
| `transport` | ev_charging_network | Modest all-three-capital action; tests cross-tier spillover |

Each action is placed in a county from its `applicable_counties` list, selected with a stride
to distribute across states (CO, WY, MT, NM, ID, NV). Magnitude = `unit_scale` (one full unit).

**Documented simplification:** No correlation structure is imposed across coefficients.
In reality, coefficients for related actions (e.g., wind and solar capex/employment multipliers)
likely co-vary — if wind employment coefficients are over-estimated, solar probably is too.
Modeling that correlation matrix is left as future work (requires empirical meta-analysis of
coefficient estimation errors, which is out of scope for v1). The current design therefore
overstates coefficient independence and may understate correlated-uncertainty outcomes.

In [ ]:
# ── Define representative portfolio ───────────────────────────────────────────
# (action_id, county_list_stride, rationale_tag)
# stride spreads county picks across different positions → geographic diversity
_PORTFOLIO_SPEC = [
    # Energy generation
    ('wind_utility',                   10,  'energy_generation'),
    ('solar_utility',                  20,  'energy_generation'),
    ('geothermal_utility',              0,  'energy_generation'),
    ('coal_repowering',                30,  'energy_generation'),
    ('smr_advanced',                   15,  'energy_generation'),
    ('coal_to_solar',                  40,  'energy_generation'),
    ('battery_grid',                    5,  'energy_storage'),
    # Energy transmission
    ('transmission_230kv',              0,  'energy_transmission'),
    ('microgrid',                      25,  'energy_transmission'),
    # Hydrological restoration
    ('riparian_buffer',                 0,  'hydrological_restoration'),
    ('beaver_reintroduction',          10,  'hydrological_restoration'),
    ('wetland_restoration',            20,  'hydrological_restoration'),
    ('watershed_protection',            5,  'hydrological_restoration'),
    ('mine_land_reclamation',          15,  'hydrological_restoration'),
    ('floodplain_reconnection',        30,  'hydrological_restoration'),
    # Terrestrial ecosystem
    ('prairie_restoration',             0,  'terrestrial_ecosystem'),
    ('invasive_treatment',             10,  'terrestrial_ecosystem'),
    ('forest_restoration',             20,  'terrestrial_ecosystem'),
    ('sagebrush_restoration',          30,  'terrestrial_ecosystem'),
    # Economic
    ('clean_manufacturing',             5,  'economic_development'),
    ('university_research_center',     15,  'settlement_social'),
    ('tribal_energy_sovereignty',       0,  'settlement_social'),
    # Social
    ('rural_broadband',                10,  'settlement_social'),
    ('health_clinic',                  20,  'settlement_social'),
    ('workforce_retraining',           30,  'settlement_social'),
    ('affordable_housing',              5,  'settlement_social'),
    ('lead_service_line',              15,  'settlement_social'),
    # Agriculture
    ('irrigation_efficiency',           0,  'agriculture'),
    ('rangeland_restoration_maintenance', 10, 'agriculture'),
    # Transport
    ('ev_charging_network',             5,  'transport'),
]

PORTFOLIO = []
_actions = BASE_LIBRARY['actions']
for action_id, stride, _tag in _PORTFOLIO_SPEC:
    if action_id not in _actions:
        print(f'  [WARN] {action_id} not in library — skipped')
        continue
    a = _actions[action_id]
    counties = a.get('applicable_counties', [])
    if not counties:
        print(f'  [WARN] {action_id} has no applicable_counties — skipped')
        continue
    geoid     = counties[stride % len(counties)]
    magnitude = a.get('unit_scale', 1000)
    PORTFOLIO.append({'action_id': action_id, 'geoid': geoid, 'magnitude': magnitude})

print(f"Portfolio: {len(PORTFOLIO)} actions")
print(f"{'Action':<35} {'GEOID':<8} {'Mag':>8}")
print("-" * 55)
for p in PORTFOLIO:
    print(f"{p['action_id']:<35} {p['geoid']:<8} {p['magnitude']:>8,}")

# ── Save portfolio ─────────────────────────────────────────────────────────────
portfolio_path = DATA_DIR / 'mc_representative_portfolio.json'
with open(portfolio_path, 'w') as f:
    json.dump({
        'created':     datetime.now(timezone.utc).isoformat(),
        'n_actions':   len(PORTFOLIO),
        'rationale':   'Fixed 30-action portfolio for MC coefficient sensitivity analysis. '
                       'Balanced across all eight action buckets and both ecological and '
                       'economic capital tiers. Held constant across all iterations; only '
                       'coefficients vary.',
        'portfolio':   PORTFOLIO,
    }, f, indent=2)
print(f"\nSaved: {portfolio_path}")

In [ ]:
# ── Enumerate perturbable coefficients ─────────────────────────────────────────
# A coefficient is perturbable iff its nominal value != 0.
# Multiplicative perturbation of 0 is always 0 regardless of factor — no information.

COEFF_KEYS = []   # list of (action_id, capital) where capital in {E, Ec, S}
COEFF_META  = {}  # (action_id, capital) → {nominal, confidence, source, bucket}

for action_id, a in BASE_LIBRARY['actions'].items():
    ees = a.get('ees_effects', {})
    conf = a.get('ees_confidence') or {}
    src  = a.get('ees_sources')   or {}
    bucket = a.get('bucket', '?')
    tier   = a.get('tier',   '?')
    for capital in ('E', 'Ec', 'S'):
        v = ees.get(capital, 0.0)
        if v == 0.0:
            continue
        key = (action_id, capital)
        COEFF_KEYS.append(key)
        COEFF_META[key] = {
            'nominal':    v,
            'confidence': conf.get(capital) if isinstance(conf, dict) else None,
            'source':     src.get(capital)  if isinstance(src,  dict) else None,
            'bucket':     bucket,
            'tier':       tier,
        }

print(f"Perturbable coefficients: {len(COEFF_KEYS)}")
print(f"  E  coefficients: {sum(1 for _, c in COEFF_KEYS if c == 'E')}")
print(f"  Ec coefficients: {sum(1 for _, c in COEFF_KEYS if c == 'Ec')}")
print(f"  S  coefficients: {sum(1 for _, c in COEFF_KEYS if c == 'S')}")

# ── Sanity check: sign preservation ───────────────────────────────────────────
negative_coeffs = [(aid, cap, meta['nominal'])
                   for (aid, cap), meta in COEFF_META.items()
                   if meta['nominal'] < 0]
print(f"\nNegative nominal coefficients: {len(negative_coeffs)}")
for aid, cap, v in negative_coeffs:
    # Factor is drawn from truncnorm on [0.5, 1.5] — always positive.
    # Multiplying a negative value by a positive factor keeps sign negative.
    worst_case = v * 1.5   # most extreme (least negative)
    assert worst_case < 0, f"Sign flip detected for {aid}/{cap}: {v} * 1.5 = {worst_case}"
    print(f"  {aid}/{cap}: nominal={v:+.4f}, perturbed range [{v*1.5:+.4f}, {v*0.5:+.4f}] — sign preserved ✓")

print("\nSign preservation assertion passed for all negative coefficients.")

## Monte Carlo Methodology

### Perturbation model
Each non-zero EES coefficient is multiplied by an independently sampled factor drawn from
a **truncated normal distribution** centered at 1.0:

```
σ = COEFF_CI_HALF_WIDTH / COEFF_CI_Z  ≈ 0.255
factor ~ TruncNormal(μ=1.0, σ=0.255, lower=0.5, upper=1.5)
```

This parameterization means the 95% confidence interval spans ±50% of the nominal coefficient
value. Truncation at [0.5, 1.5] prevents sign flips and pathological outliers.

### Documented simplifications (v1)
1. **No cross-coefficient correlation.** Each coefficient is sampled independently.
   In practice, related coefficients (e.g., wind and solar employment multipliers, or all
   hydrological restoration E effects) are likely positively correlated — if one is
   over-estimated, others probably are too. A correlated perturbation design would require
   a correlation matrix estimated from meta-analysis of coefficient error distributions,
   which is left as future work.
2. **Equal weighting of composite score.** Composite = (E + Ec + S) / 3.
   No capital-tier weights are applied; Spearman results for individual tiers are reported
   separately to compensate.
3. **Fixed portfolio.** Only one representative portfolio is evaluated as the primary subject.
   A smaller paired sub-portfolio analysis (n=1,000) is included to check ranking robustness.

In [ ]:
# ── Nominal (unperturbed) baseline run ────────────────────────────────────────
# Used as reference for sub-portfolio comparison and for the summary statement.

def apply_portfolio(base_state, portfolio, library_override=None):
    """Apply a list of portfolio entries to a state, optionally swapping the action library."""
    curr = dict(base_state)
    if library_override is not None:
        curr['action_library'] = library_override
    for entry in portfolio:
        try:
            curr, _ = te.apply_action(curr, entry['action_id'], entry['geoid'], entry['magnitude'])
        except Exception as e:
            pass
    return te.compute_ees_summary(curr)

nominal_summary = apply_portfolio(BASE_STATE, PORTFOLIO)
nom = nominal_summary['study_area']
NOM_E, NOM_Ec, NOM_S = nom['E'], nom['Ec'], nom['S']
NOM_COMPOSITE = (NOM_E + NOM_Ec + NOM_S) / 3.0

base_nom = te.compute_ees_summary(BASE_STATE)['study_area']
print("Baseline EES (no portfolio):")
print(f"  E={base_nom['E']:.4f}  Ec={base_nom['Ec']:.4f}  S={base_nom['S']:.4f}  composite={(base_nom['E']+base_nom['Ec']+base_nom['S'])/3:.4f}")
print("\nNominal portfolio EES (unperturbed coefficients):")
print(f"  E={NOM_E:.4f}  Ec={NOM_Ec:.4f}  S={NOM_S:.4f}  composite={NOM_COMPOSITE:.4f}")
print(f"\nPortfolio delta (nominal − baseline):")
print(f"  ΔE={NOM_E-base_nom['E']:+.4f}  ΔEc={NOM_Ec-base_nom['Ec']:+.4f}  ΔS={NOM_S-base_nom['S']:+.4f}")

In [ ]:
# ── Single-iteration runner (used in both quick-test and full-run modes) ───────
# This inline version is used for QUICK_TEST (avoids subprocess overhead).
# The full run uses mc_worker.run_iteration via ProcessPoolExecutor.

from scipy.stats import truncnorm as _truncnorm

# Pre-compute truncnorm clip points (same math as mc_worker.py)
_A_CLIP = (0.5 - 1.0) / SIGMA
_B_CLIP = (1.5 - 1.0) / SIGMA
_BASE_ACTIONS_FROZEN = copy.deepcopy(BASE_STATE['action_library']['actions'])

def _run_one_inline(iter_idx: int, seed: int) -> dict:
    """Inline single-process iteration (no globals, all captured via closure)."""
    rng = np.random.default_rng(int(seed))
    rv  = _truncnorm(_A_CLIP, _B_CLIP, loc=1.0, scale=SIGMA)
    raw_factors = rv.rvs(size=len(COEFF_KEYS), random_state=rng)

    perturbed_actions = copy.deepcopy(_BASE_ACTIONS_FROZEN)
    factor_dict: dict = {}

    for i, (action_id, capital) in enumerate(COEFF_KEYS):
        orig = _BASE_ACTIONS_FROZEN.get(action_id, {}).get('ees_effects', {}).get(capital, 0.0)
        if orig == 0.0:
            factor_dict[(action_id, capital)] = 1.0
            continue
        f = float(raw_factors[i])
        factor_dict[(action_id, capital)] = f
        if action_id in perturbed_actions:
            perturbed_actions[action_id].setdefault('ees_effects', {})[capital] = orig * f

    perturbed_library = dict(BASE_STATE['action_library'])
    perturbed_library['actions'] = perturbed_actions

    curr = dict(BASE_STATE)
    curr['action_library'] = perturbed_library

    for entry in PORTFOLIO:
        try:
            curr, _ = te.apply_action(curr, entry['action_id'], entry['geoid'], entry['magnitude'])
        except Exception:
            pass

    sa = te.compute_ees_summary(curr)['study_area']
    E, Ec, S = sa['E'], sa['Ec'], sa['S']
    composite = (E + Ec + S) / 3.0

    row = {'iter_idx': int(iter_idx), 'seed': int(seed),
           'E': E, 'Ec': Ec, 'S': S, 'composite': composite}
    for (aid, cap), fval in factor_dict.items():
        row[f'f__{aid}__{cap}'] = fval
    return row

print("_run_one_inline defined. Test (1 iteration)...")
t0 = time.time()
test_row = _run_one_inline(0, 42)
print(f"  iter=0 seed=42  E={test_row['E']:.4f} Ec={test_row['Ec']:.4f} S={test_row['S']:.4f} composite={test_row['composite']:.4f}")
print(f"  Factor columns: {sum(1 for k in test_row if k.startswith('f__'))}")
print(f"  Time: {time.time()-t0:.2f}s per iteration (× {N_ITER} = ~{(time.time()-t0)*N_ITER/60:.1f} min single-threaded)")

In [ ]:
# ── Checkpoint detection & resume ─────────────────────────────────────────────
# On restart: detect existing checkpoint and skip already-completed iterations.

completed_indices = set()
existing_rows = []

if DO_CHECKPOINT and CHECKPOINT_PATH.exists():
    existing_df = pd.read_parquet(CHECKPOINT_PATH)
    completed_indices = set(existing_df['iter_idx'].astype(int).tolist())
    existing_rows = existing_df.to_dict('records')
    print(f"Checkpoint found: {len(completed_indices)} / {N_ITER} iterations already complete.")
    if len(completed_indices) >= N_ITER:
        print("  → Run is already complete. Skip to analysis cells.")
    else:
        print(f"  → Resuming from iteration {max(completed_indices)+1 if completed_indices else 0}.")
else:
    if not DO_CHECKPOINT:
        print("QUICK_TEST mode — checkpointing disabled.")
    else:
        print("No checkpoint found — starting fresh run.")

# Determine which indices still need to run
all_indices = list(range(N_ITER))
pending_indices = [i for i in all_indices if i not in completed_indices]
print(f"Pending iterations: {len(pending_indices)}")

In [ ]:
# ── Main simulation loop ───────────────────────────────────────────────────────
#
# QUICK_TEST=True  →  single-process inline loop (simple, no subprocess overhead)
# QUICK_TEST=False →  ProcessPoolExecutor with mc_worker.py
#                     (survives kernel crashes via parquet checkpoint)

# Seed generation: reproducible per iteration regardless of run order
master_rng = np.random.default_rng(2025_07_22)   # fixed master seed
ALL_SEEDS  = master_rng.integers(0, 2**32, size=N_ITER)

all_rows = list(existing_rows)  # may already contain checkpoint results

if not pending_indices:
    print("Nothing to run — all iterations complete.")

elif QUICK_TEST:
    # ── Quick test: single-process ───────────────────────────────────────────
    print(f"Running {len(pending_indices)} iterations (single-process quick test)...")
    t0 = time.time()
    for i, idx in enumerate(pending_indices):
        row = _run_one_inline(idx, ALL_SEEDS[idx])
        all_rows.append(row)
        if (i + 1) % max(1, len(pending_indices) // 10) == 0 or (i + 1) == len(pending_indices):
            elapsed  = time.time() - t0
            remain   = elapsed / (i + 1) * (len(pending_indices) - i - 1)
            print(f"  {i+1:>4}/{len(pending_indices)}  elapsed={elapsed:.0f}s  est_remain={remain:.0f}s")
    print(f"Done. Total: {time.time()-t0:.1f}s")

else:
    # ── Full run: ProcessPoolExecutor ─────────────────────────────────────────
    # Workers import mc_worker.py from src/; set PYTHONPATH so spawned processes find it.
    os.environ['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')

    import mc_worker

    _portfolio_json   = json.dumps(PORTFOLIO)
    _coeff_keys_json  = json.dumps(COEFF_KEYS)
    _initargs         = (_portfolio_json, _coeff_keys_json, COEFF_CI_HALF_WIDTH, COEFF_CI_Z)

    print(f"Launching {N_WORKERS} workers for {len(pending_indices)} iterations…")
    t0 = time.time()
    buffer   = []   # accumulates rows between checkpoint writes
    n_done   = 0

    task_args = [(int(idx), int(ALL_SEEDS[idx])) for idx in pending_indices]

    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        initializer=mc_worker.worker_init,
        initargs=_initargs,
    ) as executor:
        futures = {executor.submit(mc_worker.run_iteration, a): a[0] for a in task_args}

        for fut in as_completed(futures):
            try:
                row = fut.result()
            except Exception as exc:
                print(f"  [ERR] iter {futures[fut]}: {exc}")
                continue

            all_rows.append(row)
            buffer.append(row)
            n_done += 1

            # Progress log every CHECKPOINT_INTERVAL iterations
            if n_done % CHECKPOINT_INTERVAL == 0 or n_done == len(pending_indices):
                elapsed = time.time() - t0
                total_done = len(completed_indices) + n_done
                rate   = n_done / elapsed if elapsed > 0 else 1
                remain = (N_ITER - total_done) / rate
                print(f"  iteration {total_done:,}/{N_ITER:,}  "
                      f"{elapsed/60:.1f}m elapsed  "
                      f"est. {remain/60:.1f}m remaining")

                # Write checkpoint
                if buffer:
                    ckpt_df = pd.DataFrame(all_rows)
                    ckpt_df.to_parquet(CHECKPOINT_PATH, index=False)
                    buffer.clear()
                    print(f"  ✓ Checkpoint written ({total_done:,} rows)")

    print(f"Full run complete. Total wall-clock: {(time.time()-t0)/60:.1f} min")

print(f"\nTotal rows collected: {len(all_rows)}")

In [ ]:
# ── Consolidate results DataFrame ──────────────────────────────────────────────
MC = pd.DataFrame(all_rows).sort_values('iter_idx').reset_index(drop=True)

IS_PARTIAL = len(MC) < N_ITER
if IS_PARTIAL:
    print(f"⚠  PARTIAL RESULTS: {len(MC):,} / {N_ITER:,} iterations complete.")
    print("   Analysis below is valid but based on incomplete sample.")
else:
    print(f"✓ Complete: {len(MC):,} iterations")

print(f"\nColumns: {len(MC.columns)} total ({sum(1 for c in MC.columns if c.startswith('f__'))} factor columns)")
print(MC[['E','Ec','S','composite']].describe().round(4))

In [ ]:
# ── 4.1  Distribution summary ──────────────────────────────────────────────────

def dist_summary(series, label):
    return {
        'metric':   label,
        'mean':     series.mean(),
        'median':   series.median(),
        'sd':       series.std(),
        'p5':       series.quantile(0.05),
        'p95':      series.quantile(0.95),
        'nominal':  {'E': NOM_E, 'Ec': NOM_Ec, 'S': NOM_S, 'composite': NOM_COMPOSITE}.get(label, float('nan')),
    }

dist_rows = [
    dist_summary(MC['E'],         'E'),
    dist_summary(MC['Ec'],        'Ec'),
    dist_summary(MC['S'],         'S'),
    dist_summary(MC['composite'], 'composite'),
]
dist_df = pd.DataFrame(dist_rows).set_index('metric').round(4)
print("Distribution of EES scores across all MC iterations:")
display(dist_df)

# Plain-language summary
comp = dist_df.loc['composite']
print()
print(f"Under current coefficient uncertainty (±{COEFF_CI_HALF_WIDTH*100:.0f}% at "
      f"{COEFF_CI_Z:.2f}σ / 95% CI), the composite resilience score for the "
      f"representative portfolio is {comp['mean']:.4f} ± {comp['sd']:.4f} "
      f"(90% interval: [{comp['p5']:.4f}, {comp['p95']:.4f}]).")
print(f"Nominal (unperturbed) composite: {NOM_COMPOSITE:.4f}")
print(f"Coefficient uncertainty adds ±{comp['sd']/NOM_COMPOSITE*100:.1f}% "
      f"relative uncertainty to the composite score.")
label_str = '(PARTIAL SAMPLE)' if IS_PARTIAL else ''
print(f"Sample size: {len(MC):,} iterations {label_str}")

In [ ]:
# ── 4.2  Spearman rank correlations ───────────────────────────────────────────
# For each perturbable coefficient, compute its Spearman correlation with each
# outcome metric (composite, E, Ec, S).

factor_cols = [c for c in MC.columns if c.startswith('f__')]

print(f"Computing Spearman correlations for {len(factor_cols)} factor columns × 4 outcomes…")
t0 = time.time()

# Only compute where factor has variance (skip always-1.0 columns)
varied_cols = [c for c in factor_cols if MC[c].std() > 1e-9]
print(f"  {len(varied_cols)} columns have non-zero variance (rest are trivially corr=0)")

spearman_rows = []
for col in varied_cols:
    # Parse action_id and capital from column name f__{action_id}__{capital}
    parts  = col[3:].rsplit('__', 1)   # strip 'f__', split on last '__'
    action_id = parts[0]
    capital   = parts[1]
    key        = (action_id, capital)
    meta       = COEFF_META.get(key, {})

    vals = MC[col].values
    corr_composite, _ = spearmanr(vals, MC['composite'].values)
    corr_E,         _ = spearmanr(vals, MC['E'].values)
    corr_Ec,        _ = spearmanr(vals, MC['Ec'].values)
    corr_S,         _ = spearmanr(vals, MC['S'].values)

    spearman_rows.append({
        'action_id':          action_id,
        'capital':            capital,
        'column':             col,
        'nominal':            meta.get('nominal', float('nan')),
        'bucket':             meta.get('bucket', '?'),
        'tier':               meta.get('tier', '?'),
        'confidence':         meta.get('confidence'),
        'source':             meta.get('source'),
        'spearman_composite': corr_composite,
        'spearman_E':         corr_E,
        'spearman_Ec':        corr_Ec,
        'spearman_S':         corr_S,
        'abs_composite':      abs(corr_composite),
    })

CORR_DF = pd.DataFrame(spearman_rows).sort_values('abs_composite', ascending=False).reset_index(drop=True)
print(f"Done in {time.time()-t0:.1f}s")
print("\nTop 10 coefficients by |Spearman(factor, composite)|:")
display(CORR_DF[['action_id','capital','nominal','bucket','spearman_composite','spearman_E','spearman_Ec','spearman_S','confidence']].head(10))

In [ ]:
# ── 4.2  Tornado charts ────────────────────────────────────────────────────────

def _tornado(corr_df, score_col, title, outpath, top_n=20):
    """Horizontal tornado chart: top_n coefficients by |Spearman| for score_col."""
    # Select top_n by absolute value of the target score column
    df = corr_df.copy()
    df['_abs_sel'] = df[score_col].abs()
    df = df.nlargest(top_n, '_abs_sel').drop('_abs_sel', axis=1)
    df = df.sort_values(score_col, ascending=True)  # ascending → largest bar at top

    labels = [f"{r['action_id']}\n({r['capital']})" for _, r in df.iterrows()]
    values = df[score_col].values
    colors = ['#e74c3c' if v < 0 else '#2980b9' for v in values]

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.45)))
    ax.barh(range(len(labels)), values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel(f'Spearman ρ with {score_col}', fontsize=10)
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlim(-1, 1)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    n_label = f'(n={len(MC):,}{" — partial" if IS_PARTIAL else ""})'
    ax.text(0.98, 0.02, n_label, transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color='gray')
    fig.tight_layout()
    fig.savefig(outpath, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved: {outpath}")


# One tornado per outcome (composite, E, Ec, S)
for outcome, col_name in [('composite', 'spearman_composite'),
                           ('E',         'spearman_E'),
                           ('Ec',        'spearman_Ec'),
                           ('S',         'spearman_S')]:
    _tornado(
        CORR_DF,
        col_name,
        f'Tornado: Top 20 Coefficient Sensitivities → {outcome} score',
        FIG_DIR / f'mc_tornado_{outcome}.png',
        top_n=20,
    )

print("All tornado charts saved.")

In [ ]:
# ── 4.3  Sub-portfolio ranking robustness check ────────────────────────────────
#
# Three sub-portfolios (subsets of the representative portfolio) are evaluated
# under the SAME coefficient draws as the main run (paired samples).
# Question: how often does coefficient uncertainty flip the ranking?
#
# Scope: limited to min(SUB_PORTFOLIO_N, N_ITER) paired draws for speed.

# Define sub-portfolios as subsets by bucket focus
_ENV_IDS  = {'riparian_buffer','beaver_reintroduction','wetland_restoration',
              'watershed_protection','mine_land_reclamation','floodplain_reconnection',
              'prairie_restoration','invasive_treatment','forest_restoration','sagebrush_restoration'}
_ECON_IDS = {'wind_utility','solar_utility','geothermal_utility','coal_repowering',
              'smr_advanced','coal_to_solar','battery_grid','transmission_230kv',
              'clean_manufacturing','university_research_center'}
_BAL_IDS  = {'wind_utility','prairie_restoration','riparian_buffer','rural_broadband',
              'health_clinic','clean_manufacturing','irrigation_efficiency',
              'battery_grid','workforce_retraining','forest_restoration'}

env_portfolio  = [p for p in PORTFOLIO if p['action_id'] in _ENV_IDS]
econ_portfolio = [p for p in PORTFOLIO if p['action_id'] in _ECON_IDS]
bal_portfolio  = [p for p in PORTFOLIO if p['action_id'] in _BAL_IDS]

print(f"Sub-portfolio sizes: env={len(env_portfolio)}  econ={len(econ_portfolio)}  balanced={len(bal_portfolio)}")

# Nominal composite for each sub-portfolio (unperturbed coefficients)
def _nominal_composite(ptf):
    s = apply_portfolio(BASE_STATE, ptf)
    sa = s['study_area']
    return (sa['E'] + sa['Ec'] + sa['S']) / 3.0

nom_env  = _nominal_composite(env_portfolio)
nom_econ = _nominal_composite(econ_portfolio)
nom_bal  = _nominal_composite(bal_portfolio)
print(f"Nominal composites → env={nom_env:.4f}  econ={nom_econ:.4f}  balanced={nom_bal:.4f}")
nom_rank = sorted(['env','econ','balanced'], key=lambda x: {'env':nom_env,'econ':nom_econ,'balanced':nom_bal}[x], reverse=True)
print(f"Nominal ranking: {nom_rank}")

# Paired evaluation using stored seeds
n_sub = min(SUB_PORTFOLIO_N, len(MC))
print(f"\nEvaluating sub-portfolios over {n_sub} paired draws (single-process)…")
t0 = time.time()

rank_flips = 0
sub_results = []

for i in range(n_sub):
    seed = int(MC.iloc[i]['seed'])
    rng  = np.random.default_rng(seed)
    rv   = _truncnorm(_A_CLIP, _B_CLIP, loc=1.0, scale=SIGMA)
    raw_f = rv.rvs(size=len(COEFF_KEYS), random_state=rng)

    # Build perturbed actions (same as main loop)
    pa = copy.deepcopy(_BASE_ACTIONS_FROZEN)
    for j, (action_id, capital) in enumerate(COEFF_KEYS):
        orig = _BASE_ACTIONS_FROZEN.get(action_id, {}).get('ees_effects', {}).get(capital, 0.0)
        if orig != 0.0:
            pa[action_id].setdefault('ees_effects', {})[capital] = orig * float(raw_f[j])

    pl = dict(BASE_STATE['action_library'])
    pl['actions'] = pa

    def _comp(ptf):
        s = apply_portfolio(BASE_STATE, ptf, library_override=pl)
        sa = s['study_area']
        return (sa['E'] + sa['Ec'] + sa['S']) / 3.0

    c_env  = _comp(env_portfolio)
    c_econ = _comp(econ_portfolio)
    c_bal  = _comp(bal_portfolio)

    this_rank = sorted(['env','econ','balanced'],
                       key=lambda x: {'env':c_env,'econ':c_econ,'balanced':c_bal}[x], reverse=True)
    if this_rank != nom_rank:
        rank_flips += 1

    sub_results.append({'seed': seed, 'c_env': c_env, 'c_econ': c_econ, 'c_bal': c_bal,
                        'flip': this_rank != nom_rank})

    if (i + 1) % max(1, n_sub // 5) == 0:
        print(f"  {i+1}/{n_sub}  flips so far: {rank_flips} ({rank_flips/(i+1)*100:.1f}%)")

print(f"Done in {time.time()-t0:.1f}s")

sub_df = pd.DataFrame(sub_results)
flip_rate = rank_flips / n_sub
print(f"\nSub-portfolio ranking robustness ({n_sub} paired draws):")
print(f"  Nominal ranking: {nom_rank}")
print(f"  Rank flips:      {rank_flips} / {n_sub} = {flip_rate*100:.1f}%")
print()
if flip_rate < 0.05:
    print("  → Ranking is ROBUST: coefficient uncertainty flips the portfolio ranking in "
          f"fewer than 5% of draws. Decision is unlikely to change with better coefficients.")
elif flip_rate < 0.25:
    print("  → Ranking is MODERATELY SENSITIVE: portfolio choice could change in "
          f"{flip_rate*100:.0f}% of scenarios. High-leverage coefficients warrant validation.")
else:
    print(f"  → Ranking is HIGHLY SENSITIVE ({flip_rate*100:.0f}% flip rate). Coefficient "
          "uncertainty materially affects which portfolio looks best.")

print(f"\n  [Note: sub-portfolio analysis uses {n_sub} draws vs. {len(MC)} main run draws.]")

In [ ]:
# ── 4.4  Low-confidence flag cross-reference ───────────────────────────────────
# Join sensitivity ranking with confidence tier to produce the validation-priority table.
# This identifies: which high-leverage coefficients are weakly sourced?

# Confidence tier mapping (for display)
_CONF_ORDER = {'high': 0, 'medium': 1, 'low': 2, 'proxy': 3, 'expert_estimate': 3, None: 4}

TOP_N_PRIORITY = 15

priority_df = CORR_DF.head(TOP_N_PRIORITY)[[
    'action_id', 'capital', 'bucket', 'nominal',
    'spearman_composite', 'confidence', 'source'
]].copy()
priority_df.columns = [
    'action_id', 'indicator_field', 'capital_tier',
    'nominal_coeff', 'spearman_corr_composite',
    'current_confidence', 'current_source'
]
priority_df.insert(0, 'rank', range(1, TOP_N_PRIORITY + 1))

# Flag high-leverage + low-confidence coefficients
priority_df['HIGH_LEVERAGE_LOW_CONF'] = (
    priority_df['current_confidence'].isin([None, 'low', 'proxy', 'expert_estimate'])
)

print(f"Top {TOP_N_PRIORITY} coefficients by Spearman correlation with composite score:")
display(priority_df)

n_flagged = priority_df['HIGH_LEVERAGE_LOW_CONF'].sum()
print(f"\n⚑ {n_flagged} of the top-{TOP_N_PRIORITY} most sensitive coefficients are "
      f"currently low-confidence or unsourced.")
if n_flagged:
    flagged = priority_df[priority_df['HIGH_LEVERAGE_LOW_CONF']]
    print("  These are highest-priority for empirical validation:")
    for _, r in flagged.iterrows():
        print(f"    rank {r['rank']:>2}: {r['action_id']}/{r['indicator_field']}  "
              f"ρ={r['spearman_corr_composite']:.3f}  conf={r['current_confidence']}")

# Save validation priority CSV
prio_path = DATA_DIR / 'mc_validation_priority.csv'
priority_df.to_csv(prio_path, index=False)
print(f"\nSaved: {prio_path}")

In [ ]:
# ── Update network_metadata.json ───────────────────────────────────────────────
# Create or update with an mc_sensitivity block.

meta_path = DATA_DIR / 'network_metadata.json'
if meta_path.exists():
    with open(meta_path) as f:
        metadata = json.load(f)
else:
    metadata = {
        'created':       datetime.now(timezone.utc).isoformat(),
        'schema_version': schema_ver,
        'action_library': str(lib_path.name),
    }

top5 = CORR_DF.head(5)[['action_id','capital','spearman_composite']].to_dict('records')
metadata['mc_sensitivity'] = {
    'run_date':                    datetime.now(timezone.utc).isoformat(),
    'notebook':                    '15_coefficient_monte_carlo.ipynb',
    'quick_test':                  QUICK_TEST,
    'n_iterations_requested':      N_ITER,
    'n_iterations_completed':      len(MC),
    'is_partial':                  IS_PARTIAL,
    'coeff_ci_half_width':         COEFF_CI_HALF_WIDTH,
    'coeff_ci_z':                  COEFF_CI_Z,
    'sigma':                       round(SIGMA, 4),
    'n_perturbable_coefficients':  len(COEFF_KEYS),
    'composite_mean':              round(MC['composite'].mean(), 4),
    'composite_sd':                round(MC['composite'].std(), 4),
    'composite_p5':                round(MC['composite'].quantile(0.05), 4),
    'composite_p95':               round(MC['composite'].quantile(0.95), 4),
    'nominal_composite':           round(NOM_COMPOSITE, 4),
    'top5_coefficients_by_sensitivity': top5,
    'n_high_leverage_low_confidence':   int(n_flagged),
    'sub_portfolio_flip_rate':          round(flip_rate, 4),
    'sub_portfolio_n_draws':            n_sub,
}

with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Updated: {meta_path}")
print(json.dumps(metadata['mc_sensitivity'], indent=2))

## Final Notes

### Interpreting the tornado charts
A positive Spearman ρ means that when this coefficient is drawn high (closer to 1.5×), the
composite score tends to be higher.  Negative ρ means the coefficient has a *drag* role
(e.g., a cost/harm coefficient that enters negatively).  The **magnitude** of ρ, not its sign,
determines leverage for validation purposes.

### What to do with the validation-priority table
1. Start with the highest-ranked low-confidence coefficients.
2. For each: find the original source in `ees_sources`, assess quality, and seek an
   empirical estimate (literature meta-analysis, expert elicitation, or regional data pull).
3. Updating even one high-leverage, low-confidence coefficient with a tighter empirical
   estimate will reduce composite-score uncertainty more than updating any lower-ranked
   coefficient regardless of its confidence tier.

### Re-running after coefficient updates
- Update `mw_action_library_v3.json` with revised coefficient and confidence fields.
- Delete or rename `mc_results_checkpoint.parquet`.
- Re-run this notebook with `QUICK_TEST = False`.